In [ ]:
!unzip CIFAR-10-images-master.zip

Archive:  CIFAR-10-images-master.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of CIFAR-10-images-master.zip or
        CIFAR-10-images-master.zip.zip, and cannot find CIFAR-10-images-master.zip.ZIP, period.


In [ ]:
import os
import torch
import imageio.v2 as imageio
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score, f1_score


In [ ]:
raw_data_train = '/content/CIFAR-10-images-master/train/'
raw_data_test  = '/content/CIFAR-10-images-master/test/'

In [ ]:
transform_train = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor()
])

transform_test = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor()
])

In [ ]:
dataset_train = []
labels_train  = []
targets_train = []

for folder in sorted(os.listdir(raw_data_train)):
    for image in os.listdir(os.path.join(raw_data_train, folder)):
        if folder not in labels_train:
            labels_train.append(folder)
        targets_train.append(labels_train.index(folder))

        img_arr = imageio.imread(os.path.join(raw_data_train, folder, image), pilmode="RGB")
        img = transform_train(img_arr)
        dataset_train.append(img)

data_train = torch.stack(dataset_train)
targets_train = torch.tensor(targets_train).type(torch.LongTensor)


FileNotFoundError: [Errno 2] No such file or directory: '/content/CIFAR-10-images-master/train/'

In [ ]:
dataset_test = []
labels_test = []
targets_test = []

for folder in sorted(os.listdir(raw_data_test)):
    for image in os.listdir(os.path.join(raw_data_test, folder)):
        if folder not in labels_test:
            labels_test.append(folder)
        targets_test.append(labels_test.index(folder))

        img_arr = imageio.imread(os.path.join(raw_data_test, folder, image), pilmode="RGB")
        img = transform_test(img_arr)
        dataset_test.append(img)

In [ ]:
data_test = torch.stack(dataset_test)
targets_test = torch.tensor(targets_test).type(torch.LongTensor)

print(f"Loaded {len(data_train)} training images and {len(data_test)} test images.")

Loaded 50000 training images and 10000 test images.


In [ ]:
CIFAR_train_list = [(data_train[i], targets_train[i].item()) for i in range(data_train.shape[0])]
CIFAR_test_list  = [(data_test[i], targets_test[i].item()) for i in range(data_test.shape[0])]


In [ ]:
batch_size = 128

In [ ]:
train_dl = DataLoader(CIFAR_train_list, batch_size=batch_size, shuffle=True)
test_dl  = DataLoader(CIFAR_test_list, batch_size=1000, shuffle=False)

In [ ]:
class CNN_net(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(128 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [ ]:
model = CNN_net()
opt = optim.Adam(model.parameters(), lr=0.0007)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
def training_loop(N_Epochs, model, loss_fn, opt):
    print("Starting training...")
    for epoch in range(N_Epochs):
        model.train()
        for xb, yb in train_dl:
            y_pred = model(xb)
            loss = loss_fn(y_pred, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if epoch % 5 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


In [ ]:
def print_metrics_function(y_true, y_pred):
    print('-' * 30)
    print(f'Accuracy: {accuracy_score(y_true, y_pred):.2f}')
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(f'Precision: {precision_score(y_true, y_pred, average="weighted"):.3f}')
    print(f'Recall: {recall_score(y_true, y_pred, average="weighted"):.3f}')
    print(f'F1-measure: {f1_score(y_true, y_pred, average="weighted"):.3f}')
    print('-' * 30)

In [ ]:
training_loop(40, model, loss_fn, opt)

Starting training...
Epoch 0, Loss: 0.1674
Epoch 5, Loss: 0.1740
Epoch 10, Loss: 0.1696
Epoch 15, Loss: 0.1545
Epoch 20, Loss: 0.1533
Epoch 25, Loss: 0.1373
Epoch 30, Loss: 0.1648


In [ ]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for x_real, y_real in test_dl:
        y_pred = model(x_real)
        _, indices = torch.max(y_pred, dim=1)
        all_preds.extend(indices.numpy())
        all_targets.extend(y_real.numpy())

print_metrics_function(all_targets, all_preds)

------------------------------
Accuracy: 0.70
Confusion Matrix:
[[820  21  55  21   8   8   5   6  39  17]
 [ 32 828   7  15   1   2   8   3  31  73]
 [ 76   6 651  61  61  70  34  25  11   5]
 [ 39   4 101 501  30 234  32  32  12  15]
 [ 41   2 127  83 560  68  29  79   9   2]
 [ 18   3  73 148  27 658  12  49   7   5]
 [ 18   2  81  78  38  56 702   7  11   7]
 [ 33   2  39  38  53  76   6 738   0  15]
 [111  36  15  24   1   7   4   3 778  21]
 [ 55  98  12  24   4  12   3  14  24 754]]
Precision: 0.710
Recall: 0.699
F1-measure: 0.701
------------------------------
